In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

aisehack_theme_1_path = kagglehub.competition_download('aisehack-theme-1')

print('Data source import complete.')


In [ ]:
!pip install -q terratorch "numpy<2.0.0"

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import sys
import os
import sys
import numpy as np
import torch

import terratorch
from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets.transforms import FlattenTemporalIntoChannels, UnflattenTemporalFromChannels

import albumentations as A
from albumentations.pytorch import ToTensorV2

import lightning.pytorch as pl
from lightning.pytorch.loggers import TensorBoardLogger
from lightning.pytorch.callbacks import ModelCheckpoint
import rasterio

# --- 1. Define Kaggle Paths ---
DATASET_PATH = '/kaggle/input/competitions/aisehack-theme-1/data'
OUT_DIR = '/kaggle/working/'

# train_transform =A.Compose([
#         A.D4(), # Random flips and rotation
#         A.pytorch.transforms.ToTensorV2(),
#     ]),
train_transform = A.Compose([
    A.HorizontalFlip(p=0.4),
    A.VerticalFlip(p=0.7),
    A.RandomRotate90(p=0.6),
    A.Transpose(p=0.6),
    ToTensorV2()  # <--- CRITICAL: Converts NumPy to PyTorch Tensor
])

# --- 2. Initialize DataModule ---
datamodule = terratorch.datamodules.GenericNonGeoSegmentationDataModule(
    batch_size=2, # T4x2 can usually handle a batch size of 8 or 16 here
    num_workers=2,
    num_classes=2,

    # Define dataset paths
    train_data_root=os.path.join(DATASET_PATH, 'image'),
    train_label_data_root=os.path.join(DATASET_PATH, 'label'),
    val_data_root=os.path.join(DATASET_PATH, 'image'),
    val_label_data_root=os.path.join(DATASET_PATH, 'label'),
    test_data_root=os.path.join(DATASET_PATH, 'image'),
    test_label_data_root=os.path.join(DATASET_PATH, 'label'),

    # Define splits
    train_split=os.path.join(DATASET_PATH, 'split/train.txt'),
    val_split=os.path.join(DATASET_PATH, 'split/val.txt'),
    test_split=os.path.join(DATASET_PATH, 'split/test.txt'),

    # Adjust grep to catch standard .tif files
    img_grep='*image.tif',
    label_grep='*label.tif',

    # Data Augmentation (Crucial for small datasets!)
    train_transform = train_transform,
    val_transform=None,
    test_transform=None,

    # Global means and stds for the 6 bands (calculated across the whole dataset)
    means=[841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551],
    stds=[473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894],

    no_data_replace=0,
    no_label_replace=-1,
    predict_data_root="/kaggle/input/competitions/aisehack-theme-1/data/prediction/image"
)

# Setup train and val datasets internally
datamodule.setup("fit")

print(f"Training patches: {len(datamodule.train_dataset)}")
print(f"Validation patches: {len(datamodule.val_dataset)}")
print("DataModule is configured and ready!")

In [ ]:
train_dataset = datamodule.train_dataset
print(len(train_dataset))

# checking datasets validation split size
val_dataset = datamodule.val_dataset
len(val_dataset)

In [ ]:
# --- 1. Hyperparameters ---
EPOCHS = 35
LR = 2e-5
WEIGHT_DECAY = 0.1
HEAD_DROPOUT = 0.1 # previously 0.1
FREEZE_BACKBONE = False

BANDS = [1, 2, 3, 4, 5, 6]
NUM_FRAMES = 1
CLASS_WEIGHTS = [0.26, 0.74]
SEED = 42 # Standardized seed


import os
import random
import torch
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path
import lightning.pytorch as pl
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from lightning.pytorch.callbacks import ModelCheckpoint

def seed_everything(seed=42):
    """Ensures reproducibility across all libraries as per Rule 7.2"""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False

    pl.seed_everything(seed, workers=True)


seed_everything(42)
print(f"Environment ready. NumPy version: {np.__version__}")

# --- 2. Model Configuration ---
decoder_args = dict(
    decoder="UperNetDecoder",
    decoder_channels=256,
    decoder_scale_modules=True,
)

necks = [
    dict(
        name="ReshapeTokensToImage",
        effective_time_dim=NUM_FRAMES,
    )
]

backbone_args = dict(
    backbone_pretrained=True,
    backbone="prithvi_eo_v2_600_tl",
    backbone_bands=BANDS,
    backbone_num_frames=NUM_FRAMES,
)

model_args = dict(
    **backbone_args,
    **decoder_args,
    num_classes=2,
    head_dropout=HEAD_DROPOUT,
    necks=necks,
    rescale=True,
)

# --- 3. Initialize TerraTorch Task ---
model = SemanticSegmentationTask(
    model_args=model_args,
    plot_on_val=False,
    class_weights=CLASS_WEIGHTS,
    loss="dice",   # Swapped from focal to dice to directly optimize IoU
    lr=LR,
    optimizer="AdamW",
    optimizer_hparams=dict(weight_decay=WEIGHT_DECAY),
    ignore_index=-1,
    freeze_backbone=FREEZE_BACKBONE, # Currently False
    freeze_decoder=False,
    model_factory="EncoderDecoderFactory",
    # tiled_inference_on_testing=True,    # Fixes the testing crash
    # tiled_inference_on_validation=True, # Fixes the validation crash
    # tiled_inference_parameters=dict(
    #     h_crop=512,         # Tile height
    #     w_crop=512,         # Tile width
    #     h_stride=256,       # 50% overlap vertically to prevent seams
    #     w_stride=256,       # 50% overlap horizontally to prevent seams
    #     average_patches=True # Blends the overlapping edges perfectly
    # )
)

# --- 3.5. The Bulletproof Partial Freezing Logic ---
print("Configuring Partial Freezing for the ViT Backbone...")

# Step A: Lock the vault. Freeze every single parameter in the entire model.
for param in model.parameters():
    param.requires_grad = False

# Step B: Unfreeze the brand new UperNet Decoder and Necks
# TerraTorch/MMSeg defaults to naming these 'decode_head', 'auxiliary_head', 'neck', or 'decoder'
for name, param in model.named_parameters():
    if any(keyword in name for keyword in ["decode", "head", "neck", "decoder"]):
        param.requires_grad = True

# Step C: Unfreeze the last 4 blocks of the Prithvi ViT (Blocks 27-31)
blocks_to_unfreeze = ["blocks.28", "blocks.29", "blocks.30", "blocks.31", "norm"]
for name, param in model.named_parameters():
    if any(b in name for b in blocks_to_unfreeze):
        param.requires_grad = True

# Verify the parameter count
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")


# --- 4. Logger and Callbacks ---
logger = TensorBoardLogger(save_dir=OUT_DIR, name="prithvi_flood_logs")

checkpoint_callback = ModelCheckpoint(
    monitor="val/mIoU",
    mode="max",
    dirpath=os.path.join(OUT_DIR, "checkpoints"),
    filename="best-prithvi-{epoch:02d}-{val/mIoU:.4f}", # Matched filename to the monitor metric
    save_top_k=1,
)

# --- 5. Trainer Initialization ---
trainer = pl.Trainer(
    accelerator="gpu",
    devices=1,
    precision="16-mixed", # Corrected for T4 GPU architecture
    logger=logger,
    max_epochs=EPOCHS,
    check_val_every_n_epoch=1,
    # accumulate_grad_batches=2,
    log_every_n_steps=5,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback],
    # deterministic=True # CRITICAL for Kaggle Rule 7.2 (Reproducibility)
)

print("Model and Trainer are fully configured and ready!")

In [ ]:
trainer.fit(model, datamodule=datamodule)

In [ ]:
!ls /kaggle/working/checkpoints/

In [ ]:
!ls -lh "/kaggle/working/checkpoints/best-prithvi-epoch=22-val/"

In [ ]:
datamodule.setup("test")
#sample checkpoint from 37th epoch for testing..
best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=22-val/mIoU=0.6947.ckpt"

In [ ]:
test_results = trainer.test(model, datamodule=datamodule, ckpt_path=best_ckpt_path)

In [ ]:
import os
import torch
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from terratorch.tasks import SemanticSegmentationTask

# --- 1. SET THE PATH ---
# We use the exact path from your last output
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=09-val/loss=0.42.ckpt"
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=22-val/mIoU=0.6951.ckpt"
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=38-val/mIoU=0.6986.ckpt"
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=16-val/mIoU=0.6929.ckpt"
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=53-val/mIoU=0.6840.ckpt"
best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=22-val/mIoU=0.6947.ckpt"

print(f"Loading weights from: {best_ckpt_path}")
model = SemanticSegmentationTask.load_from_checkpoint(best_ckpt_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- 2. Setup Data ---
split_file = '/kaggle/input/competitions/aisehack-theme-1/data/split/test.txt'
image_dir = '/kaggle/input/competitions/aisehack-theme-1/data/image'
label_dir = '/kaggle/input/competitions/aisehack-theme-1/data/label'

with open(split_file, 'r') as f:
    test_names = [line.strip() for line in f.readlines()]

# Prithvi Z-Score normalization
means = np.array([841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551]).reshape(6, 1, 1)
stds = np.array([473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894]).reshape(6, 1, 1)

total_iou = 0.0
images_plotted = 0

print(f"\nEvaluating Prithvi on local test images...\n")

with torch.no_grad():
    for img_name in test_names:
        base_name = img_name[:-4] if img_name.endswith('.tif') else img_name
        img_path = os.path.join(image_dir, f"{base_name}.tif")
        if not os.path.exists(img_path): img_path = os.path.join(image_dir, f"{base_name}_image.tif")
        lbl_path = os.path.join(label_dir, f"{base_name}.tif")
        if not os.path.exists(lbl_path): lbl_path = os.path.join(label_dir, f"{base_name}_label.tif")

        with rasterio.open(img_path) as src: image = src.read().astype(np.float32)
        with rasterio.open(lbl_path) as src: label = src.read(1)

        processed_image = (image - means) / stds
        input_tensor = torch.tensor(processed_image, dtype=torch.float32).unsqueeze(0).to(device)
        label_tensor = torch.tensor(label, dtype=torch.long).to(device)

        outputs = model(input_tensor)
        prediction_logits = outputs.output
        preds_class = torch.argmax(prediction_logits, dim=1).squeeze()

        # Calculate IoU
        intersection = torch.logical_and(preds_class == 1, label_tensor == 1).float().sum()
        union = torch.logical_or(preds_class == 1, label_tensor == 1).float().sum()
        img_iou = intersection / (union + 1e-6)
        total_iou += img_iou.item()

        # Visualization
        if images_plotted < 3:
            # 1. Prepare Satellite Display (False Color NIR-R-G)
            rgb_display = image[[4, 3, 2], :, :].transpose(1, 2, 0)
            rgb_display = (rgb_display - np.nanmin(rgb_display)) / (np.nanmax(rgb_display) - np.nanmin(rgb_display) + 1e-8)
            rgb_display = np.clip(rgb_display, 0, 1)

            # 2. Prepare Binary Masks (Original high-contrast Black/White)
            pred_mask_cpu = preds_class.cpu().numpy()

            # 3. Create a High-Visibility Overlay
            # We'll use Red for the overlay so it stands out against the dark water/green land
            mask_overlay = np.zeros_like(rgb_display)
            mask_overlay[pred_mask_cpu == 1] = [1, 0, 0]  # Red for water prediction

            # Blend: Keep the background clear but add a red "glow" to predicted water
            alpha = 0.4
            blended_img = np.where(pred_mask_cpu[..., None] == 1,
                                   (rgb_display * (1-alpha) + mask_overlay * alpha),
                                   rgb_display)

            # 4. Plotting 4 Panels
            fig, axs = plt.subplots(1, 4, figsize=(20, 6))
            fig.suptitle(f"Patch: {base_name} | IoU: {img_iou.item():.4f}", fontsize=16, y=1.02)

            # Original False Color Satellite
            axs[0].imshow(rgb_display)
            axs[0].set_title("Original Satellite")
            axs[0].axis('off')

            # Binary Ground Truth (Black background, White water)
            axs[1].imshow(label, cmap='gray')
            axs[1].set_title("Ground Truth (GT)")
            axs[1].axis('off')

            # Binary Prediction (Black background, White water)
            axs[2].imshow(pred_mask_cpu, cmap='gray')
            axs[2].set_title("Prithvi Prediction")
            axs[2].axis('off')

            # The Overlay (Red predicted water on top of Original Image)
            axs[3].imshow(blended_img)
            axs[3].set_title("Detection Overlay (Red)")
            axs[3].axis('off')

            plt.tight_layout()
            plt.show()

            images_plotted += 1
avg_iou = total_iou / len(test_names)
print(f"🌍 FINAL PRITHVI TEST IoU: {avg_iou*100:.2f}%")

In [ ]:
# import lightning.pytorch as pl
# from lightning.pytorch.callbacks import ModelCheckpoint

# # 1. Define your checkpoint path
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=16-val/mIoU=0.6929.ckpt"
# # ---------------------------------------------------------
# # 🚨 THE FIX: RE-APPLY THE PADLOCKS BEFORE TRAINING
# # ---------------------------------------------------------
# print("Re-applying Partial Freezing locks before resuming...")

# # Step A: Lock everything
# for param in model.parameters():
#     param.requires_grad = False

# # Step B: Unlock the Decoder
# for name, param in model.named_parameters():
#     if any(keyword in name for keyword in ["decode", "head", "neck", "decoder"]):
#         param.requires_grad = True

# # Step C: Unlock the final 5 blocks of the 600M ViT (Blocks 27-31)
# blocks_to_unfreeze = ["blocks.27", "blocks.28", "blocks.29", "blocks.30", "blocks.31", "norm"]
# for name, param in model.named_parameters():
#     if any(b in name for b in blocks_to_unfreeze):
#         param.requires_grad = True

# # Verify the parameter count is safely back down!
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# total_params = sum(p.numel() for p in model.parameters())
# print(f"Total Parameters: {total_params:,}")
# print(f"Trainable Parameters: {trainable_params:,} ({(trainable_params/total_params)*100:.2f}%)")
# # ---------------------------------------------------------

# # 2. Set up the Callback
# checkpoint_callback = ModelCheckpoint(
#     monitor="val/mIoU",
#     mode="max",
#     dirpath="/kaggle/working/checkpoints_continued",
#     filename="best-prithvi-600M-continued-{epoch:02d}-{val/mIoU:.4f}",
#     save_top_k=1,
# )

# # 3. Initialize the Trainer
# trainer = pl.Trainer(
#     accelerator="gpu",
#     devices=1,
#     precision="16-mixed",
#     max_epochs=70,
#     callbacks=[checkpoint_callback],
#     log_every_n_steps=2,
#     check_val_every_n_epoch=1,
# )

# print(f"Resuming Prithvi 600M training from {best_ckpt_path}...")

# # 4. Launch the Training
# trainer.fit(
#     model,
#     datamodule=datamodule,
#     ckpt_path=best_ckpt_path
# )

In [ ]:
# !ls /kaggle/working/checkpoints_continued/

In [ ]:
# !ls -lh "/kaggle/working/checkpoints_continued/best-prithvi-600M-continued-epoch=40-val/"

In [ ]:
# datamodule.setup("test")
# #sample checkpoint from 37th epoch for testing..
# best_ckpt_path = "/kaggle/working/checkpoints_continued/best-prithvi-600M-continued-epoch=40-val/mIoU=0.6939.ckpt"

In [ ]:
# test_results = trainer.test(model, datamodule=datamodule, ckpt_path=best_ckpt_path)

In [ ]:
# import os
# import torch
# import rasterio
# import numpy as np
# import matplotlib.pyplot as plt
# from terratorch.tasks import SemanticSegmentationTask

# # --- 1. SET THE PATH ---
# # We use the exact path from your last output
# # best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=09-val/loss=0.42.ckpt"
# # best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=22-val/mIoU=0.6951.ckpt"
# best_ckpt_path = "/kaggle/working/checkpoints/best-prithvi-epoch=16-val/mIoU=0.6929.ckpt"

# print(f"Loading weights from: {best_ckpt_path}")
# model = SemanticSegmentationTask.load_from_checkpoint(best_ckpt_path)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model.to(device)
# model.eval()

# # --- 2. Setup Data ---
# split_file = '/kaggle/input/competitions/aisehack-theme-1/data/split/test.txt'
# image_dir = '/kaggle/input/competitions/aisehack-theme-1/data/image'
# label_dir = '/kaggle/input/competitions/aisehack-theme-1/data/label'

# with open(split_file, 'r') as f:
#     test_names = [line.strip() for line in f.readlines()]

# # Prithvi Z-Score normalization
# means = np.array([841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551]).reshape(6, 1, 1)
# stds = np.array([473.7090, 170.3611, 623.0474, 612.8465, 564.5835, 528.0894]).reshape(6, 1, 1)

# total_iou = 0.0
# images_plotted = 0

# print(f"\nEvaluating Prithvi on local test images...\n")

# with torch.no_grad():
#     for img_name in test_names:
#         base_name = img_name[:-4] if img_name.endswith('.tif') else img_name
#         img_path = os.path.join(image_dir, f"{base_name}.tif")
#         if not os.path.exists(img_path): img_path = os.path.join(image_dir, f"{base_name}_image.tif")
#         lbl_path = os.path.join(label_dir, f"{base_name}.tif")
#         if not os.path.exists(lbl_path): lbl_path = os.path.join(label_dir, f"{base_name}_label.tif")

#         with rasterio.open(img_path) as src: image = src.read().astype(np.float32)
#         with rasterio.open(lbl_path) as src: label = src.read(1)

#         processed_image = (image - means) / stds
#         input_tensor = torch.tensor(processed_image, dtype=torch.float32).unsqueeze(0).to(device)
#         label_tensor = torch.tensor(label, dtype=torch.long).to(device)

#         outputs = model(input_tensor)
#         prediction_logits = outputs.output
#         preds_class = torch.argmax(prediction_logits, dim=1).squeeze()

#         # Calculate IoU
#         intersection = torch.logical_and(preds_class == 1, label_tensor == 1).float().sum()
#         union = torch.logical_or(preds_class == 1, label_tensor == 1).float().sum()
#         img_iou = intersection / (union + 1e-6)
#         total_iou += img_iou.item()

#         # Visualization
#         if images_plotted < 3:
#             # 1. Prepare Satellite Display (False Color NIR-R-G)
#             rgb_display = image[[4, 3, 2], :, :].transpose(1, 2, 0)
#             rgb_display = (rgb_display - np.nanmin(rgb_display)) / (np.nanmax(rgb_display) - np.nanmin(rgb_display) + 1e-8)
#             rgb_display = np.clip(rgb_display, 0, 1)

#             # 2. Prepare Binary Masks (Original high-contrast Black/White)
#             pred_mask_cpu = preds_class.cpu().numpy()

#             # 3. Create a High-Visibility Overlay
#             # We'll use Red for the overlay so it stands out against the dark water/green land
#             mask_overlay = np.zeros_like(rgb_display)
#             mask_overlay[pred_mask_cpu == 1] = [1, 0, 0]  # Red for water prediction

#             # Blend: Keep the background clear but add a red "glow" to predicted water
#             alpha = 0.4
#             blended_img = np.where(pred_mask_cpu[..., None] == 1,
#                                    (rgb_display * (1-alpha) + mask_overlay * alpha),
#                                    rgb_display)

#             # 4. Plotting 4 Panels
#             fig, axs = plt.subplots(1, 4, figsize=(20, 6))
#             fig.suptitle(f"Patch: {base_name} | IoU: {img_iou.item():.4f}", fontsize=16, y=1.02)

#             # Original False Color Satellite
#             axs[0].imshow(rgb_display)
#             axs[0].set_title("Original Satellite")
#             axs[0].axis('off')

#             # Binary Ground Truth (Black background, White water)
#             axs[1].imshow(label, cmap='gray')
#             axs[1].set_title("Ground Truth (GT)")
#             axs[1].axis('off')

#             # Binary Prediction (Black background, White water)
#             axs[2].imshow(pred_mask_cpu, cmap='gray')
#             axs[2].set_title("Prithvi Prediction")
#             axs[2].axis('off')

#             # The Overlay (Red predicted water on top of Original Image)
#             axs[3].imshow(blended_img)
#             axs[3].set_title("Detection Overlay (Red)")
#             axs[3].axis('off')

#             plt.tight_layout()
#             plt.show()

#             images_plotted += 1
# avg_iou = total_iou / len(test_names)
# print(f"🌍 FINAL PRITHVI TEST IoU: {avg_iou*100:.2f}%")

In [ ]:
datamodule.setup("predict")

predictions = trainer.predict(
    model,
    datamodule=datamodule,
    ckpt_path=best_ckpt_path
)

output_dir = "/kaggle/working/prediction"
os.makedirs(output_dir, exist_ok=True)

for batch_idx, (preds, file_paths) in enumerate(predictions):

    if isinstance(preds, tuple):
        preds = preds[0]

    if preds.ndim == 4:               # [B, C, H, W]
        preds = preds.argmax(dim=1)   # [B, H, W]

    preds = preds.cpu().numpy().astype("int16")

    for i in range(preds.shape[0]):
        arr = preds[i]
        arr[arr < 0] = -1             # 🔒 enforce ignore_index

        ref_path = file_paths[i]
        with rasterio.open(ref_path) as src:
            meta = src.meta.copy()

        meta.update({
            "count": 1,
            "dtype": "int16",
            "nodata": -1,
            "compress": "lzw",
        })

        out_name = (
            os.path.splitext(os.path.basename(ref_path))[0]
            + ".tif"
        )
        out_path = os.path.join(output_dir, out_name)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(arr, 1)

        print(f"Saved {out_path}")

In [ ]:
import numpy as np

def mask_to_rle(mask):
    """
    Convert binary mask to RLE (Kaggle format).
    Mask must be 2D numpy array with values 0 or 1.
    """
    pixels = mask.flatten(order="F")  # column-major
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

In [ ]:
import rasterio
import pandas as pd
from pathlib import Path

tif_dir = Path("/kaggle/working/prediction")   # folder with .tif files
output_csv = "/kaggle/working/submission_600M_Dice_loss_Version_5_35epochs.csv"

rows = []

for tif_path in sorted(tif_dir.glob("*.tif")):
    with rasterio.open(tif_path) as src:
        mask = src.read(1)

    # Convert to binary
    mask = (mask > 0).astype(np.uint8)

    rle = mask_to_rle(mask)

    rows.append({
        "id": tif_path.name.replace("_image.tif", ""),
        "rle_mask": rle
    })

df = pd.DataFrame(rows)
df = df.replace("", 0).fillna(0) #replace null/ na with zero - kaggle compatible
df.to_csv(output_csv, index=False)
print(f"Saved Kaggle RLE CSV : {output_csv}")